# 02. 올바른 bbox로 1-stage / 2-stage YOLO 데이터셋 생성


## 목적

`01_prepare_raw_dataset.ipynb`에서 생성한 raw 이미지와 AI Hub JSON을 사용해 두 가지 YOLO 데이터셋을 동시에 만듭니다.

- 1-stage: 재질 × 오염 상태, 9개 클래스
- 2-stage detector: 재질, 3개 클래스
- AI Hub BOX `POINTS=[x, y, width, height]`를 YOLO `class x_center y_center width height`로 변환
- manifest, 변환 보고서, 무결성 검사 결과 저장

> 기존 최종 학습 폴더를 보호하기 위해 기본 출력은 `reproduced_dataset/1-stage`, `reproduced_dataset/2-stage`입니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from collections import Counter
import csv
import json
import shutil

from PIL import Image
from tqdm.auto import tqdm


## 1. 경로 및 클래스 설정


In [ ]:
TEAM_PROJECT = Path('/content/drive/MyDrive/TeamProject')
OUTPUT_ROOT = TEAM_PROJECT / 'test_dataset' / 'reproduced_dataset'
RAW_ROOT = OUTPUT_ROOT / 'raw'
STAGE_ROOTS = {
    '1-stage': OUTPUT_ROOT / '1-stage',
    '2-stage': OUTPUT_ROOT / '2-stage',
}
AUDIT_ROOT = OUTPUT_ROOT / 'dataset_build_audit'

ONE_STAGE_NAMES = [
    'can_clean', 'can_outer', 'can_inner',
    'pet_clean', 'pet_outer', 'pet_inner',
    'plastic_clean', 'plastic_outer', 'plastic_inner',
]
TWO_STAGE_NAMES = ['can', 'pet', 'plastic']

ONE_STAGE_MAP = {
    ('금속캔', '오염없음'): 0, ('금속캔', '이물질(외부)'): 1, ('금속캔', '이물질(내부)'): 2,
    ('페트병', '오염없음'): 3, ('페트병', '이물질(외부)'): 4, ('페트병', '이물질(내부)'): 5,
    ('플라스틱', '오염없음'): 6, ('플라스틱', '이물질(외부)'): 7, ('플라스틱', '이물질(내부)'): 8,
}
TWO_STAGE_MAP = {'금속캔': 0, '페트병': 1, '플라스틱': 2}
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
OVERWRITE = False

for split in ('train', 'val'):
    assert (RAW_ROOT / split / 'images').is_dir()
    assert (RAW_ROOT / split / 'labels_json').is_dir()

print('raw:', RAW_ROOT)
print('output:', STAGE_ROOTS)


## 2. 변환 함수

중요: `POINTS`의 네 값은 두 모서리 좌표가 아니라 `x, y, width, height`입니다.


In [ ]:
def normalize_dirtiness(value):
    value = '' if value is None else str(value).strip()
    aliases = {
        '외부오염': '이물질(외부)',
        '내부오염': '이물질(내부)',
        '내용물(내부)': '이물질(내부)',
    }
    return aliases.get(value, value)

def unpack_xywh(points):
    if not isinstance(points, list):
        raise ValueError('POINTS가 list가 아닙니다.')
    values = points[0] if len(points) == 1 and isinstance(points[0], list) else points
    if len(values) != 4:
        raise ValueError(f'BOX POINTS 길이가 4가 아닙니다: {values}')
    x, y, width, height = map(float, values)
    if width <= 0 or height <= 0:
        raise ValueError(f'폭 또는 높이가 0 이하입니다: {values}')
    return x, y, width, height

def to_yolo(x, y, width, height, image_width, image_height):
    x1 = max(0.0, min(x, image_width))
    y1 = max(0.0, min(y, image_height))
    x2 = max(0.0, min(x + width, image_width))
    y2 = max(0.0, min(y + height, image_height))
    if x2 <= x1 or y2 <= y1:
        raise ValueError('clipping 후 bbox 크기가 0 이하입니다.')
    return (
        ((x1 + x2) / 2) / image_width,
        ((y1 + y2) / 2) / image_height,
        (x2 - x1) / image_width,
        (y2 - y1) / image_height,
    )

def image_index(directory):
    return {p.stem: p for p in directory.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES}


## 3. 1-stage와 2-stage 데이터셋 생성


In [ ]:
for stage_root in STAGE_ROOTS.values():
    if stage_root.exists() and any(p.is_file() for p in stage_root.rglob('*')):
        if not OVERWRITE:
            raise FileExistsError(
                f'기존 결과가 있습니다: {stage_root}\n'
                '덮어쓰려면 설정 셀에서 OVERWRITE=True로 변경하세요.'
            )
        shutil.rmtree(stage_root)

for stage_root in STAGE_ROOTS.values():
    for split in ('train', 'val'):
        (stage_root / 'images' / split).mkdir(parents=True, exist_ok=True)
        (stage_root / 'labels' / split).mkdir(parents=True, exist_ok=True)
AUDIT_ROOT.mkdir(parents=True, exist_ok=True)

report = {stage: {} for stage in STAGE_ROOTS}
manifest_rows = []

for split in ('train', 'val'):
    raw_images = image_index(RAW_ROOT / split / 'images')
    json_paths = sorted((RAW_ROOT / split / 'labels_json').glob('*.json'))

    stage_stats = {
        stage: Counter(json_files=len(json_paths)) for stage in STAGE_ROOTS
    }

    for json_path in tqdm(json_paths, desc=f'{split} 변환'):
        stem = json_path.stem
        image_path = raw_images.get(stem)
        if image_path is None:
            for stats in stage_stats.values(): stats['missing_image'] += 1
            continue

        try:
            data = json.loads(json_path.read_text(encoding='utf-8'))
            image_info = data.get('IMAGE_INFO', {})
            image_width = float(image_info.get('IMAGE_WIDTH', 0))
            image_height = float(image_info.get('IMAGE_HEIGHT', 0))
            if image_width <= 0 or image_height <= 0:
                with Image.open(image_path) as image:
                    image_width, image_height = image.size
        except Exception:
            for stats in stage_stats.values(): stats['json_or_size_error'] += 1
            continue

        lines = {'1-stage': [], '2-stage': []}
        errors = []

        for ann_index, ann in enumerate(data.get('ANNOTATION_INFO', [])):
            if str(ann.get('SHAPE_TYPE', '')).strip().upper() != 'BOX':
                continue
            material = str(ann.get('CLASS', '')).strip()
            dirtiness = normalize_dirtiness(ann.get('DIRTINESS'))
            one_key = (material, dirtiness)
            if one_key not in ONE_STAGE_MAP or material not in TWO_STAGE_MAP:
                continue
            try:
                box = unpack_xywh(ann.get('POINTS'))
                xc, yc, bw, bh = to_yolo(*box, image_width, image_height)
            except Exception as exc:
                errors.append(f'annotation {ann_index}: {exc}')
                continue

            coords = f'{xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}'
            lines['1-stage'].append(f'{ONE_STAGE_MAP[one_key]} {coords}')
            lines['2-stage'].append(f'{TWO_STAGE_MAP[material]} {coords}')

        for stage, stage_root in STAGE_ROOTS.items():
            stats = stage_stats[stage]
            if not lines[stage]:
                stats['no_valid_target_box'] += 1
                continue

            shutil.copy2(image_path, stage_root / 'images' / split / image_path.name)
            (stage_root / 'labels' / split / f'{stem}.txt').write_text(
                '\n'.join(lines[stage]) + '\n', encoding='utf-8'
            )
            stats['saved_images'] += 1
            stats['saved_objects'] += len(lines[stage])
            manifest_rows.append({
                'stage': stage, 'split': split, 'stem': stem,
                'image_name': image_path.name, 'object_count': len(lines[stage]),
                'conversion_errors': ' | '.join(errors),
            })

    for stage in STAGE_ROOTS:
        report[stage][split] = dict(stage_stats[stage])

print(json.dumps(report, ensure_ascii=False, indent=2))


## 4. data.yaml 및 manifest 저장


In [ ]:
def write_yaml(stage_root, names):
    name_lines = '\n'.join(f'  {index}: {name}' for index, name in enumerate(names))
    text = (
        f'path: {stage_root}\n'
        'train: images/train\n'
        'val: images/val\n\n'
        f'names:\n{name_lines}\n'
    )
    (stage_root / 'data.yaml').write_text(text, encoding='utf-8')

write_yaml(STAGE_ROOTS['1-stage'], ONE_STAGE_NAMES)
write_yaml(STAGE_ROOTS['2-stage'], TWO_STAGE_NAMES)

with open(AUDIT_ROOT / 'dataset_manifest.csv', 'w', encoding='utf-8-sig', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=list(manifest_rows[0]))
    writer.writeheader()
    writer.writerows(manifest_rows)

with open(AUDIT_ROOT / 'conversion_report.json', 'w', encoding='utf-8') as stream:
    json.dump(report, stream, ensure_ascii=False, indent=2)

print('manifest:', AUDIT_ROOT / 'dataset_manifest.csv')


## 5. 최종 무결성 검사


In [ ]:
validation = {}

for stage, stage_root in STAGE_ROOTS.items():
    validation[stage] = {}
    train_stems = set()
    val_stems = set()

    for split in ('train', 'val'):
        images = image_index(stage_root / 'images' / split)
        labels = {p.stem: p for p in (stage_root / 'labels' / split).glob('*.txt')}
        class_counts = Counter()
        malformed = []
        object_count = 0

        for stem, label_path in labels.items():
            for line_number, line in enumerate(label_path.read_text(encoding='utf-8').splitlines(), 1):
                parts = line.split()
                try:
                    class_id = int(parts[0])
                    values = list(map(float, parts[1:5]))
                    expected_classes = len(ONE_STAGE_NAMES) if stage == '1-stage' else len(TWO_STAGE_NAMES)
                    if len(parts) != 5 or not 0 <= class_id < expected_classes:
                        raise ValueError('형식 또는 class id 오류')
                    if not all(0.0 <= value <= 1.0 for value in values):
                        raise ValueError('정규화 범위 오류')
                    if values[2] <= 0 or values[3] <= 0:
                        raise ValueError('bbox 크기 오류')
                    class_counts[class_id] += 1
                    object_count += 1
                except Exception as exc:
                    malformed.append({'file': str(label_path), 'line': line_number, 'error': str(exc)})

        stems = set(images)
        if split == 'train': train_stems = stems
        else: val_stems = stems

        validation[stage][split] = {
            'images': len(images), 'labels': len(labels), 'objects': object_count,
            'missing_labels': sorted(set(images) - set(labels)),
            'missing_images': sorted(set(labels) - set(images)),
            'malformed': malformed,
            'class_counts': {str(k): v for k, v in sorted(class_counts.items())},
        }

    validation[stage]['train_val_stem_overlap'] = len(train_stems & val_stems)

for stage, result in validation.items():
    for split in ('train', 'val'):
        assert not result[split]['missing_labels']
        assert not result[split]['missing_images']
        assert not result[split]['malformed']
    assert result['train_val_stem_overlap'] == 0

with open(AUDIT_ROOT / 'dataset_validation.json', 'w', encoding='utf-8') as stream:
    json.dump(validation, stream, ensure_ascii=False, indent=2)

print(json.dumps(validation, ensure_ascii=False, indent=2))
print('검증 통과')
